# Glassware Canonical Product Analysis

Phase 3 workflow for `Laboratory Glassware`:

1. Pull labeled subset from `PROPOSED_L5` + `PRODUCTS_LCG`
2. Normalize naming/attributes
3. Build canonical identity key
4. Select representative records
5. Assign canonical product IDs
6. Export canonical and membership tables

This notebook emphasizes deterministic, reproducible analytics with optional LLM assistance only for naming/refinement.

In [1]:
import json
import re
from hashlib import sha1
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import get_snowflake_session

In [2]:
# ------------------
# Config
# ------------------
DB = "SNOWFLAKE_LEARNING_DB"
SCHEMA = "SMCMAHON_PRODUCTS"

SOURCE_PRODUCTS_TABLE = f"{DB}.{SCHEMA}.PRODUCTS_LCG"
SOURCE_LABELS_TABLE = f"{DB}.{SCHEMA}.PROPOSED_L5"
TARGET_LABELS = [
    "Laboratory Bottles",
    "Caps and Closures",
    "Laboratory Glassware",
    "Tubes and Vials",
    "Pyrex Glassware",
    "Chemical Safety",
    "Chromatography Columns",
]

# Output options
OUT_DIR = PROJECT_ROOT / "artifacts" / "analysis" / "glassware_canonical"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_TO_SNOWFLAKE = False
CANONICAL_TABLE = f"{DB}.{SCHEMA}.CANONICAL_GLASSWARE_PRODUCTS"
MEMBERSHIP_TABLE = f"{DB}.{SCHEMA}.CANONICAL_GLASSWARE_MEMBERS"

# Core-attribute parsing behavior
REQUIRE_CONTAINER_TYPE = True
MIN_GROUP_SIZE_FOR_REVIEW = 2

# Source column aliases (case-insensitive) -> standardized working columns
SOURCE_COLUMN_ALIASES = {
    "PRODUCT_ID": ["product_id", "PRODUCT_ID"],
    "PRODUCT_NAME": ["product_name", "PRODUCT_NAME"],
    "DESCRIPTION": ["description", "DESCRIPTION"],
    "PRICING_STATUS_C": ["pricing_status_c", "PRICING_STATUS_C"],
    "LIST_PRICE_C": ["list_price_c", "LIST_PRICE_C"],
    "SPECIFICATION_ASSIGNMENTS": [
        "specification_assignments_c",
        "SPECIFICATION_ASSIGNMENTS_C",
        "specification_assignments",
        "SPECIFICATION_ASSIGNMENTS",
    ],
}

In [3]:
# ------------------
# Extract glassware-labeled records
# ------------------
sf = get_snowflake_session()
sf.sql(f"USE DATABASE {DB}").collect()
sf.sql(f"USE SCHEMA {SCHEMA}").collect()

label_list_sql = ", ".join(["'" + str(lbl).replace("'", "''") + "'" for lbl in TARGET_LABELS])
query = f"""
SELECT
    p.*,
    l.PROPOSED_CLUSTER,
    l.PROPOSED_LABEL
FROM {SOURCE_LABELS_TABLE} l
JOIN {SOURCE_PRODUCTS_TABLE} p
  ON p.PRODUCT_ID = l.PRODUCT_ID
WHERE l.PROPOSED_LABEL IN ({label_list_sql})
  AND UPPER(COALESCE(p.DESCRIPTION, '')) NOT LIKE '%INSERT%'
"""

df_raw = sf.sql(query).to_pandas()
print(f"Rows loaded: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns)}")

# Build lower-case lookup for flexible column mapping.
col_lookup = {c.lower(): c for c in df_raw.columns}

def resolve_col(candidates: list[str]) -> str | None:
    for c in candidates:
        if c.lower() in col_lookup:
            return col_lookup[c.lower()]
    return None

std_df = pd.DataFrame(index=df_raw.index)
for std_col, aliases in SOURCE_COLUMN_ALIASES.items():
    src_col = resolve_col(aliases)
    if src_col is not None:
        std_df[std_col] = df_raw[src_col]

# Always retain proposed outputs from label table if present.
for c in ["PROPOSED_CLUSTER", "PROPOSED_LABEL"]:
    src = resolve_col([c])
    if src is not None:
        std_df[c] = df_raw[src]

required_std = ["PRODUCT_ID", "PRODUCT_NAME", "DESCRIPTION", "SPECIFICATION_ASSIGNMENTS"]
missing_std = [c for c in required_std if c not in std_df.columns]
if missing_std:
    raise ValueError(f"Missing required standardized columns: {missing_std}")

df = std_df.copy()
print("Standardized columns:", sorted(df.columns))

# ------------------
# Dedupe before analysis (one row per PRODUCT_ID)
# ------------------
if "PRODUCT_ID" not in df.columns:
    raise ValueError("Cannot dedupe: PRODUCT_ID is missing")

rows_before_dedupe = len(df)

# Prefer rows with richer text/spec content when duplicates exist.
_df = df.copy()
_df["_has_desc"] = _df["DESCRIPTION"].fillna("").astype(str).str.len() > 0
_df["_has_spec"] = _df["SPECIFICATION_ASSIGNMENTS"].fillna("").astype(str).str.len() > 0
_df["_label_present"] = _df["PROPOSED_LABEL"].notna() if "PROPOSED_LABEL" in _df.columns else False

_df = _df.sort_values(
    by=["PRODUCT_ID", "_label_present", "_has_desc", "_has_spec"],
    ascending=[True, False, False, False],
)

dupe_rows = _df[_df.duplicated(subset=["PRODUCT_ID"], keep=False)].copy()
conflicting_label_products = 0
if not dupe_rows.empty and "PROPOSED_LABEL" in dupe_rows.columns:
    conflicting_label_products = int(
        dupe_rows.groupby("PRODUCT_ID")["PROPOSED_LABEL"].nunique(dropna=True).gt(1).sum()
    )

df = _df.drop_duplicates(subset=["PRODUCT_ID"], keep="first").drop(
    columns=["_has_desc", "_has_spec", "_label_present"], errors="ignore"
)

rows_after_dedupe = len(df)
print(f"Rows before dedupe: {rows_before_dedupe:,}")
print(f"Rows after dedupe:  {rows_after_dedupe:,}")
print(f"Duplicate rows removed: {rows_before_dedupe - rows_after_dedupe:,}")
print(f"Products with conflicting labels across duplicates: {conflicting_label_products:,}")

df.head(3)

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZLdb5swFMX%2FFeQ9g02WTytJlTSLFq1JUfMxbW8u3BAXsAnXBNq%2FfiYfU%2FfQSntD5hz%2Fju%2B5w7s6S50TFCi1GhHfY8QBFepIqnhEtpu52ycOGqEikWoFI%2FIKSO7GQxRZmvNJaQ7qCY4loHHsRQp582NEykJxLVAiVyID5Cbk68nygbc8xgUiFMbiyNUSobSsgzE5p7SqKq%2F66ukipi3GGGUDalWN5At5h8g%2FZ%2BSFNjrU6c1S2zd9gPApazcIq7CE4GqcSnUZwWeU54sI%2BffNJnCDx%2FWGOJPb6%2B61wjKDYg3FSYawfXq4BECbIE%2FCTq%2Fd6XsluiDQuL6HSlf7VCQQ6iwvjb3Ws190DxFNdSztsBazEckTGR0XzzHsWPbjOFnF38rlSjy8pD8Hj%2Bvjdu635a%2FllJ1emAqC3TIkzu5WbaupdoFYwkI1hRp7xFpdl7Vd5m9Yi3e6nPW9fn%2FwmzgzW6hUwpydt9QYSjskgDo8CBWDpxMjziFFntO%2F%2BSnUiXmT9TGrlz3oJdG0O%2BhRRE2b3shldfg5SDH%2B74EM6Xv7dQ1XtpnFLNCpDF%2BduS4yYT4uzvf884mM3P1ZyiETMp1EUQGItsA01dV9AcLYbTdFCYSOL9R%2F9338Bw%3D%3D&RelayState=ver%3A3-hint%3A9376132675904534-ETMsDgAAAZ1G%2BES5ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEHuU%2FRda2g4B9bHw6l35Z

,PRODUCT_ID,PRODUCT_NAME,DESCRIPTION,PRICING_STATUS_C,LIST_PRICE_C,SPECIFICATION_ASSIGNMENTS,PROPOSED_CLUSTER,PROPOSED_LABEL
34174,01tPK00000GbCx8YAF,"Empty Bottle with Trigger Sprayer, Graduated, ...",Product Size: 6PC | Size Quantity: 1/CS | Manu...,Priced,30.55,"{""Product Size"":""6PC"",""Size Quantity"":""1/CS"",""...",2,Laboratory Bottles
51340,01tPK00000GbCx9YAF,"Empty Bottle with Trigger Sprayer, Graduated, ...",Product Size: 6PC | Size Quantity: 1/CS | Manu...,Priced,26.71,"{""Product Size"":""6PC"",""Size Quantity"":""1/CS"",""...",2,Laboratory Bottles
29773,01tPK00000GbCxAYAV,"Rectangular tip, polypropylene handle",Product Size: 500PC | Size Quantity: 1/CS | Ma...,Priced,254.51,"{""Product Size"":""500PC"",""Size Quantity"":""1/CS""...",16,Caps and Closures


In [4]:
# ------------------
# QA: label coverage in this run scope
# ------------------
label_counts = (
    df["PROPOSED_LABEL"]
    .fillna("<NULL>")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("PROPOSED_LABEL")
    .reset_index(name="ROW_COUNT")
)
label_counts["PCT_OF_SCOPE"] = (100.0 * label_counts["ROW_COUNT"] / max(len(df), 1)).round(2)

print(f"Rows in scope: {len(df):,}")
print(f"Distinct labels in scope: {label_counts.shape[0]}")
display(label_counts)

missing_targets = sorted(set(TARGET_LABELS) - set(label_counts["PROPOSED_LABEL"].tolist()))
if missing_targets:
    print("Missing target labels from this pull:", missing_targets)
else:
    print("All TARGET_LABELS present in this pull.")

Rows in scope: 79,011
Distinct labels in scope: 7


,PROPOSED_LABEL,ROW_COUNT,PCT_OF_SCOPE
0,Laboratory Glassware,14815,18.75
1,Tubes and Vials,14298,18.10
2,Laboratory Bottles,13605,17.22
3,Caps and Closures,12894,16.32
4,Chemical Safety,12833,16.24
5,Pyrex Glassware,6989,8.85
6,Chromatography Columns,3577,4.53


All TARGET_LABELS present in this pull.


In [5]:
# ------------------
# Normalization helpers
# ------------------
CONTAINER_TYPES = [
    "beaker", "flask", "bottle", "vial", "tube", "jar", "cylinder", "pipette", "buret", "dish", "funnel",
    "column", "cartridge", "cap", "closure", "stopper", "septum", "liner", "insert", "plug", "sprayer",
    "reservoir", "pouch", "bag", "box", "boat", "cup", "filter", "rack",
]

# Subtypes capture shape/form inside a broad family.
SUBTYPE_PATTERNS = [
    "boston round",
    "french square",
    "erlenmeyer",
    "volumetric",
    "round bottom",
    "straight sided round",
    "wide mouth",
    "narrow mouth",
    "graduated",
    "desiccator",
    "precision point",
    "amber",
    "clear",
    "reagent",
    "media",
    "dropper",
    "screw top",
    "snap cap",
    "crimp top",
    "serum",
    "headspace",
]

MATERIAL_KEYWORDS = {
    "borosilicate": "borosilicate_glass",
    "pyrex": "borosilicate_glass",
    "quartz": "quartz_glass",
    "soda lime": "soda_lime_glass",
    "ptfe": "ptfe",
    "polypropylene": "polypropylene",
    "polyethylene": "polyethylene",
    "amber": "amber_glass",
    "glass": "glass_unspecified",
}

TARGET_LABEL_TO_DOMAIN = {
    "Laboratory Bottles": "bottleware",
    "Caps and Closures": "closures",
    "Laboratory Glassware": "glassware",
    "Tubes and Vials": "tubes_vials",
    "Pyrex Glassware": "glassware",
    "Chemical Safety": "safety",
    "Chromatography Columns": "chromatography",
}

VOL_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(ml|mL|l|L|ul|uL|µl|µL)")
DIM_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(mm|cm|in|\")")
PACK_RE = re.compile(r"(?:pack|pk|case|box|qty|quantity|size quantity)\s*[:x\- ]\s*(\d+)", re.IGNORECASE)

# Key normalization for inconsistent specification_assignments JSON keys.
SPEC_KEY_MAP = {
    "product size": "size_value",
    "size": "size_value",
    "capacity": "size_value",
    "volume": "size_value",
    "size quantity": "size_quantity",
    "qty": "size_quantity",
    "quantity": "size_quantity",
    "material": "material",
    "type": "container_type",
    "product type": "container_type",
    "closure type": "closure_type",
    "cap type": "closure_type",
    "column type": "chromatography_type",
}


def normalize_text(s: str) -> str:
    s = str(s or "")
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def parse_spec_json(value) -> dict:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    if isinstance(value, dict):
        return value
    txt = str(value).strip()
    if not txt:
        return {}
    try:
        parsed = json.loads(txt)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def normalize_spec_keys(spec: dict) -> dict:
    out = {}
    for k, v in spec.items():
        nk = normalize_text(k).replace("_", " ")
        nk = re.sub(r"\s+", " ", nk)
        std_k = SPEC_KEY_MAP.get(nk, nk)
        out[std_k] = v
    return out


def detect_container_type(text: str, spec_norm: dict | None = None) -> str | None:
    if spec_norm:
        st = normalize_text(spec_norm.get("container_type", ""))
        if st:
            for t in CONTAINER_TYPES:
                if t in st:
                    return t
    for t in CONTAINER_TYPES:
        if re.search(rf"\b{re.escape(t)}s?\b", text):
            return t
    return None


def detect_material(text: str, spec_norm: dict | None = None) -> str | None:
    if spec_norm:
        sm = normalize_text(spec_norm.get("material", ""))
        for k, v in MATERIAL_KEYWORDS.items():
            if k in sm:
                return v
    for k, v in MATERIAL_KEYWORDS.items():
        if k in text:
            return v
    return None


def detect_container_subtype(text: str, spec_norm: dict | None = None) -> str | None:
    low = normalize_text(text)
    if spec_norm:
        low = f"{low} {normalize_text(spec_norm.get('container_type', ''))}".strip()
    for pat in SUBTYPE_PATTERNS:
        if pat in low:
            return pat
    return None


def infer_primary_domain(text: str, proposed_label: str | None = None) -> str:
    label = str(proposed_label or "")
    if label in TARGET_LABEL_TO_DOMAIN:
        return TARGET_LABEL_TO_DOMAIN[label]
    low = normalize_text(text)
    if any(k in low for k in ["cap", "closure", "stopper", "sept", "liner", "plug"]):
        return "closures"
    if any(k in low for k in ["spill", "hazmat", "safety", "eyewash", "absorbent"]):
        return "safety"
    if any(k in low for k in ["chromatography", "hplc", "guard column", "spe", "prepcolumn", "cartridge"]):
        return "chromatography"
    return "glassware"


def detect_closure_style(text: str, spec_norm: dict | None = None) -> str | None:
    low = normalize_text(text)
    if spec_norm:
        low = f"{low} {normalize_text(spec_norm.get('closure_type', ''))}".strip()
    styles = [
        "screw", "crimp", "snap", "push", "plug", "stopper", "septum", "linerless", "lined", "vented",
    ]
    for s in styles:
        if s in low:
            return s
    return None


def detect_safety_type(text: str) -> str | None:
    low = normalize_text(text)
    patterns = {
        "spill_kit": ["spill kit", "spill control"],
        "absorbent": ["absorbent", "sorbent", "pad", "sock", "pillow"],
        "disinfectant": ["decon", "isopropyl", "ethanol", "sanitizer", "wipe"],
        "waste_handling": ["waste", "hazmat", "disposal"],
    }
    for name, kws in patterns.items():
        if any(k in low for k in kws):
            return name
    return None


def detect_chromatography_type(text: str, spec_norm: dict | None = None) -> str | None:
    low = normalize_text(text)
    if spec_norm:
        low = f"{low} {normalize_text(spec_norm.get('chromatography_type', ''))}".strip()
    patterns = {
        "hplc_column": ["hplc", "uhplc"],
        "gc_column": ["gc column", "gas chromatography"],
        "guard_column": ["guard column"],
        "prep_column": ["prepcolumn", "prep column"],
        "spe_cartridge": ["spe", "solid phase extraction", "cartridge"],
        "ion_exchange": ["ion exchange"],
        "c18": [" c18", "c-18"],
    }
    for name, kws in patterns.items():
        if any(k in low for k in kws):
            return name
    if "column" in low:
        return "column_other"
    return None


def parse_primary_volume_ml(text: str, spec_norm: dict | None = None) -> float | None:
    spec_candidates = []
    if spec_norm:
        spec_candidates.append(str(spec_norm.get("size_value", "")))
    spec_candidates.append(text)

    for cand in spec_candidates:
        m = VOL_RE.search(str(cand))
        if not m:
            continue
        val = float(m.group(1))
        unit = m.group(2).lower()
        if unit == "l":
            return val * 1000.0
        if unit in {"ul", "µl"}:
            return val / 1000.0
        return val
    return None


def parse_primary_dimension_mm(text: str) -> float | None:
    m = DIM_RE.search(text)
    if not m:
        return None
    val = float(m.group(1))
    unit = m.group(2).lower()
    if unit == "cm":
        return val * 10.0
    if unit in {'in', '"'}:
        return val * 25.4
    return val


def parse_pack_count(text: str, spec_norm: dict | None = None) -> int | None:
    if spec_norm:
        sq = normalize_text(spec_norm.get("size_quantity", ""))
        m2 = re.search(r"(\d+)", sq)
        if m2:
            return int(m2.group(1))
    m = PACK_RE.search(text)
    if m:
        return int(m.group(1))
    return None

In [6]:
# ------------------
# Feature extraction for canonicalization
# ------------------
name_text = df.get("PRODUCT_NAME", "").fillna("").astype(str)
desc_text = df.get("DESCRIPTION", "").fillna("").astype(str)
combined_text = (name_text + " " + desc_text).map(normalize_text)

df["NORM_TEXT"] = combined_text

df["SPEC_PARSED"] = df["SPECIFICATION_ASSIGNMENTS"].map(parse_spec_json)
df["SPEC_NORM"] = df["SPEC_PARSED"].map(normalize_spec_keys)
df["SPEC_SIZE_VALUE"] = df["SPEC_NORM"].map(lambda d: d.get("size_value") if isinstance(d, dict) else None)
df["SPEC_SIZE_QUANTITY"] = df["SPEC_NORM"].map(lambda d: d.get("size_quantity") if isinstance(d, dict) else None)

df["CONTAINER_FAMILY"] = [detect_container_type(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["CONTAINER_SUBTYPE"] = [detect_container_subtype(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
# Backward-compatible alias used in downstream cells.
df["CONTAINER_TYPE"] = df["CONTAINER_FAMILY"]

df["PRIMARY_DOMAIN"] = [
    infer_primary_domain(t, lbl) for t, lbl in zip(df["NORM_TEXT"], df.get("PROPOSED_LABEL", pd.Series(index=df.index)))
]

df["MATERIAL_STD"] = [detect_material(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["VOLUME_ML"] = [parse_primary_volume_ml(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["DIMENSION_MM"] = df["NORM_TEXT"].map(parse_primary_dimension_mm)
df["PACK_COUNT"] = [parse_pack_count(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["CLOSURE_STYLE"] = [detect_closure_style(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["SAFETY_TYPE"] = df["NORM_TEXT"].map(detect_safety_type)
df["CHROMATOGRAPHY_TYPE"] = [detect_chromatography_type(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]

# Basic completeness score favors records useful as canonical exemplars.
score_cols = [
    "PRIMARY_DOMAIN",
    "CONTAINER_FAMILY",
    "CONTAINER_SUBTYPE",
    "MATERIAL_STD",
    "CLOSURE_STYLE",
    "SAFETY_TYPE",
    "CHROMATOGRAPHY_TYPE",
    "VOLUME_ML",
    "DIMENSION_MM",
    "PACK_COUNT",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "PRODUCT_NAME",
    "DESCRIPTION",
]
df["COMPLETENESS_SCORE"] = sum(df[c].notna().astype(int) for c in score_cols if c in df.columns)

df[[
    "PRODUCT_ID",
    "PROPOSED_LABEL",
    "PRIMARY_DOMAIN",
    "CONTAINER_TYPE",
    "CONTAINER_SUBTYPE",
    "CLOSURE_STYLE",
    "SAFETY_TYPE",
    "CHROMATOGRAPHY_TYPE",
    "MATERIAL_STD",
    "VOLUME_ML",
    "DIMENSION_MM",
    "PACK_COUNT",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "COMPLETENESS_SCORE",
]].head(10)

,PRODUCT_ID,PROPOSED_LABEL,PRIMARY_DOMAIN,CONTAINER_TYPE,CONTAINER_SUBTYPE,CLOSURE_STYLE,SAFETY_TYPE,CHROMATOGRAPHY_TYPE,MATERIAL_STD,VOLUME_ML,DIMENSION_MM,PACK_COUNT,SPEC_SIZE_VALUE,SPEC_SIZE_QUANTITY,COMPLETENESS_SCORE
34174,01tPK00000GbCx8YAF,Laboratory Bottles,bottleware,bottle,graduated,None,None,None,None,NaN,NaN,1.0,6PC,1/CS,8
51340,01tPK00000GbCx9YAF,Laboratory Bottles,bottleware,bottle,graduated,None,None,None,None,NaN,NaN,1.0,6PC,1/CS,8
29773,01tPK00000GbCxAYAV,Caps and Closures,closures,None,None,None,None,None,polypropylene,NaN,NaN,1.0,500PC,1/CS,7
23747,01tPK00000GbCxBYAV,Laboratory Bottles,bottleware,bottle,None,None,None,None,None,NaN,NaN,1.0,12PC,1/CS,7
58298,01tPK00000GbCxCYAV,Laboratory Bottles,bottleware,bottle,None,None,disinfectant,None,None,NaN,NaN,1.0,4PC,1/CS,8
14322,01tPK00000GbCxIYAV,Pyrex Glassware,glassware,flask,volumetric,stopper,None,None,borosilicate_glass,50.0,NaN,12.0,1 PC,12/CS,11
58290,01tPK00000GbCxJYAV,Pyrex Glassware,glassware,flask,volumetric,stopper,None,None,borosilicate_glass,500.0,NaN,12.0,1 PC,12/CS,11
30060,01tPK00000GbCxKYAV,Pyrex Glassware,glassware,flask,volumetric,screw,None,None,borosilicate_glass,100.0,NaN,12.0,1 PC,12/CS,11
48724,01tPK00000GbCxLYAV,Pyrex Glassware,glassware,flask,volumetric,screw,None,None,borosilicate_glass,NaN,NaN,12.0,1 PC,12/CS,10
64824,01tPK00000GbCxMYAV,Pyrex Glassware,glassware,flask,volumetric,screw,None,None,borosilicate_glass,25.0,NaN,12.0,1 PC,12/CS,11


In [7]:
# ------------------
# Build canonical identity key + IDs
# ------------------

def canonical_key(row: pd.Series) -> str:
    domain = row.get("PRIMARY_DOMAIN") or "unknown"
    family = row.get("CONTAINER_FAMILY") or "unknown"
    subtype = row.get("CONTAINER_SUBTYPE") or "unknown"
    mat = row.get("MATERIAL_STD") or "unknown"

    vol = row.get("VOLUME_ML")
    vol_bucket = "na" if pd.isna(vol) else str(int(round(float(vol))))

    dim = row.get("DIMENSION_MM")
    dim_bucket = "na" if pd.isna(dim) else str(int(round(float(dim))))

    # Use normalized size text as backup discriminator when numeric parsing is unavailable.
    size_txt = normalize_text(row.get("SPEC_SIZE_VALUE", ""))
    size_bucket = size_txt[:40] if size_txt else "na"

    closure = normalize_text(row.get("CLOSURE_STYLE", ""))[:20] or "na"
    safety = normalize_text(row.get("SAFETY_TYPE", ""))[:20] or "na"
    chrom = normalize_text(row.get("CHROMATOGRAPHY_TYPE", ""))[:20] or "na"

    # pack count is variant, not core identity.
    return "|".join([domain, family, subtype, mat, vol_bucket, dim_bucket, size_bucket, closure, safety, chrom])


df["CANONICAL_KEY"] = df.apply(canonical_key, axis=1)

if REQUIRE_CONTAINER_TYPE:
    unknown_mask = df["CONTAINER_TYPE"].isna()
    print(f"Rows missing container type: {int(unknown_mask.sum()):,}")

# Stable canonical product ID from canonical key
unique_keys = sorted(df["CANONICAL_KEY"].dropna().unique())
key_to_id = {k: f"CGW_{sha1(k.encode('utf-8')).hexdigest()[:10].upper()}" for k in unique_keys}
df["CANONICAL_PRODUCT_ID"] = df["CANONICAL_KEY"].map(key_to_id)

print(f"Unique canonical groups: {df['CANONICAL_PRODUCT_ID'].nunique():,}")
df[["PRODUCT_ID", "CANONICAL_KEY", "CANONICAL_PRODUCT_ID"]].head(10)

Rows missing container type: 18,492
Unique canonical groups: 21,702


,PRODUCT_ID,CANONICAL_KEY,CANONICAL_PRODUCT_ID
34174,01tPK00000GbCx8YAF,bottleware|bottle|graduated|unknown|na|na|6pc|...,CGW_F8B27C6DDC
51340,01tPK00000GbCx9YAF,bottleware|bottle|graduated|unknown|na|na|6pc|...,CGW_F8B27C6DDC
29773,01tPK00000GbCxAYAV,closures|unknown|unknown|polypropylene|na|na|5...,CGW_A934B2F4AE
23747,01tPK00000GbCxBYAV,bottleware|bottle|unknown|unknown|na|na|12pc|n...,CGW_E635C988F0
58298,01tPK00000GbCxCYAV,bottleware|bottle|unknown|unknown|na|na|4pc|na...,CGW_C6A3671F17
14322,01tPK00000GbCxIYAV,glassware|flask|volumetric|borosilicate_glass|...,CGW_1B77CD4DCA
58290,01tPK00000GbCxJYAV,glassware|flask|volumetric|borosilicate_glass|...,CGW_C96BDB000E
30060,01tPK00000GbCxKYAV,glassware|flask|volumetric|borosilicate_glass|...,CGW_BD6F316DAA
48724,01tPK00000GbCxLYAV,glassware|flask|volumetric|borosilicate_glass|...,CGW_A768F2C1A1
64824,01tPK00000GbCxMYAV,glassware|flask|volumetric|borosilicate_glass|...,CGW_3961002CD1


In [8]:
# ------------------
# Select representative record per canonical group
# ------------------

sort_cols = ["CANONICAL_PRODUCT_ID", "COMPLETENESS_SCORE"]
rep_idx = (
    df.sort_values(sort_cols, ascending=[True, False])
    .groupby("CANONICAL_PRODUCT_ID", as_index=False)
    .head(1)
    .index
)

canonical_df = df.loc[rep_idx].copy()
canonical_df = canonical_df.rename(columns={
    "PRODUCT_ID": "REPRESENTATIVE_PRODUCT_ID",
    "PRODUCT_NAME": "CANONICAL_PRODUCT_NAME",
    "DESCRIPTION": "CANONICAL_DESCRIPTION",
})

# Group size and basic confidence signals
group_sizes = df.groupby("CANONICAL_PRODUCT_ID").size().rename("GROUP_SIZE")
canonical_df = canonical_df.merge(group_sizes, on="CANONICAL_PRODUCT_ID", how="left")
canonical_df["NEEDS_REVIEW"] = canonical_df["GROUP_SIZE"] < MIN_GROUP_SIZE_FOR_REVIEW

canonical_cols = [
    "CANONICAL_PRODUCT_ID",
    "CANONICAL_KEY",
    "REPRESENTATIVE_PRODUCT_ID",
    "PROPOSED_LABEL",
    "PRIMARY_DOMAIN",
    "CANONICAL_PRODUCT_NAME",
    "CANONICAL_DESCRIPTION",
    "CONTAINER_FAMILY",
    "CONTAINER_SUBTYPE",
    "CLOSURE_STYLE",
    "SAFETY_TYPE",
    "CHROMATOGRAPHY_TYPE",
    "MATERIAL_STD",
    "VOLUME_ML",
    "DIMENSION_MM",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "GROUP_SIZE",
    "NEEDS_REVIEW",
]
canonical_cols = [c for c in canonical_cols if c in canonical_df.columns]
canonical_df = canonical_df[canonical_cols].sort_values(["GROUP_SIZE", "CANONICAL_PRODUCT_ID"], ascending=[False, True])

membership_df = df[[
    "PRODUCT_ID",
    "CANONICAL_PRODUCT_ID",
    "CANONICAL_KEY",
    "PROPOSED_CLUSTER",
    "PROPOSED_LABEL",
    "PRIMARY_DOMAIN",
    "CONTAINER_FAMILY",
    "CONTAINER_SUBTYPE",
    "CLOSURE_STYLE",
    "SAFETY_TYPE",
    "CHROMATOGRAPHY_TYPE",
    "PACK_COUNT",
    "COMPLETENESS_SCORE",
]].copy()

print("Canonical table rows:", len(canonical_df))
print("Membership table rows:", len(membership_df))
canonical_df.head(10)

Canonical table rows: 21702
Membership table rows: 79011


,CANONICAL_PRODUCT_ID,CANONICAL_KEY,REPRESENTATIVE_PRODUCT_ID,PROPOSED_LABEL,PRIMARY_DOMAIN,CANONICAL_PRODUCT_NAME,CANONICAL_DESCRIPTION,CONTAINER_FAMILY,CONTAINER_SUBTYPE,CLOSURE_STYLE,SAFETY_TYPE,CHROMATOGRAPHY_TYPE,MATERIAL_STD,VOLUME_ML,DIMENSION_MM,SPEC_SIZE_VALUE,SPEC_SIZE_QUANTITY,GROUP_SIZE,NEEDS_REVIEW
21497,CGW_FD3BBD9554,closures|unknown|unknown|unknown|na|na|1 pc|na...,01tPK00000GbQQqYAN,Caps and Closures,closures,Protecting cover Head (MSE),Product Size: 1 PC | Size Quantity: 1-EA,None,None,None,None,None,None,NaN,NaN,1 PC,1-EA,818,False
20894,CGW_F659F44066,safety|unknown|unknown|unknown|na|na|na|na|was...,01tPK00000HdtW0YAJ,Chemical Safety,safety,"PCR sealing film, Adhesive, Pk.100, CS500",Size: | Additional Shipping for Hazmat and o...,None,None,None,waste_handling,None,None,NaN,NaN,,None,780,False
913,CGW_0A8B12939B,closures|cap|unknown|unknown|na|na|1 pc|na|na|na,01tPK00000GbQRhYAN,Caps and Closures,closures,transport lock cap (2 pieces),Product Size: 1 PC | Size Quantity: 1-EA,cap,None,None,None,None,None,NaN,NaN,1 PC,1-EA,509,False
19635,CGW_E7AA54E4AE,tubes_vials|unknown|unknown|unknown|na|na|1 pc...,01tPK00000GbXbNYAV,Tubes and Vials,tubes_vials,CROSSTEX DUO-CHECK STERILIZATION POUCHES,Product Size: 1 PC | Size Quantity: 20/CS | Ma...,None,None,None,None,None,None,NaN,NaN,1 PC,20/CS,327,False
17547,CGW_CED32B8D90,chromatography|unknown|unknown|unknown|na|5|1 ...,01tPK00000GbNcfYAF,Chromatography Columns,chromatography,ODS-HYPERSIL-2 3UM 50X4.6MM,Product Size: 1 PC | Size Quantity: 1-EA,None,None,None,None,None,None,NaN,4.6,1 PC,1-EA,288,False
20279,CGW_EE9A50F22F,safety|tube|unknown|unknown|2|na|na|na|waste_h...,01tPK00000Hdk7TYAR,Chemical Safety,safety,"1.5mL Flat Top Microcentrifuge Tube, Red (C003...",Size: | Additional Shipping for Hazmat and o...,tube,None,None,waste_handling,None,None,1.5,NaN,,None,286,False
19311,CGW_E3E7FE7013,safety|tube|unknown|unknown|na|na|na|na|waste_...,01tPK00000HdtS0YAJ,Chemical Safety,safety,Cryobox 81 Tubes 3.6-4.5,Size: | Additional Shipping for Hazmat and o...,tube,None,None,waste_handling,None,None,NaN,NaN,,None,284,False
3300,CGW_267B884537,chromatography|unknown|unknown|unknown|na|na|1...,01tPK00000GbNcuYAF,Chromatography Columns,chromatography,HYPERSIL PHENYL-2 5UM,Product Size: 1 PC | Size Quantity: 1-EA,None,None,None,None,None,None,NaN,NaN,1 PC,1-EA,278,False
14834,CGW_AEE3ECE078,chromatography|unknown|unknown|unknown|na|2|1 ...,01tPK00000GbNd0YAF,Chromatography Columns,chromatography,HYPERCARB 3UM 10X2.1MM,Product Size: 1 PC | Size Quantity: 2/PK,None,None,None,None,None,None,NaN,2.1,1 PC,2/PK,239,False
18477,CGW_D9ACD85D3F,closures|cap|unknown|unknown|na|9|1 pc|na|na|na,01tPK00000GcO05YAF,Caps and Closures,closures,CAP SCR 9MM,Product Size: 1 PC | Size Quantity: 100/PK,cap,None,None,None,None,None,NaN,9.0,1 PC,100/PK,223,False


## Unknown Container-Type Review (run before save/persist)

Use these cells to inspect `UNKNOWN` container types and add targeted synonym rules.

Suggested loop:
1. Run unknown profiling cells
2. Add/adjust `CONTAINER_TYPE_SYNONYMS`
3. Rerun the reclassification cell
4. Rerun canonical ID + representative selection cells
5. Then run save/persist cells

In [9]:
# Profile unknown container-type records
unk = df[df["CONTAINER_TYPE"].isna()].copy()
print(f"Unknown rows: {len(unk):,} ({len(unk)/max(len(df),1):.1%})")

if len(unk) > 0:
    unk["PRODUCT_NAME_NORM"] = unk["PRODUCT_NAME"].fillna("").str.lower().str.strip()
    display(
        unk["PRODUCT_NAME_NORM"]
        .value_counts()
        .head(50)
        .rename_axis("product_name")
        .reset_index(name="count")
    )

    # Token frequency from unknown records
    text = (unk["PRODUCT_NAME"].fillna("") + " " + unk["DESCRIPTION"].fillna("")).str.lower()
    tokens = text.str.replace(r"[^a-z0-9 ]", " ", regex=True).str.split()

    from collections import Counter

    ctr = Counter()
    for row in tokens:
        ctr.update([t for t in row if len(t) >= 3])

    top_tokens_df = pd.DataFrame(ctr.most_common(100), columns=["token", "count"])
    display(top_tokens_df.head(40))
else:
    print("No unknowns to profile.")

Unknown rows: 18,492 (23.4%)


,product_name,count
0,insulation rpo-p,56
1,5ml macrotubes,22
2,asp global safe-t-fill capillary blood collect...,21
3,seal,21
4,hypersil gold aq 1.9um,18
5,hypersil gold pfp 5um,16
6,crosstex duo-check sterilization pouches,16
7,surestrain premium cell strainers,16
8,hypersil green pah 3um,16
9,hypersil gold pfp 3um,16


,token,count
0,size,30446
1,product,17720
2,manufacturer,13791
3,number,13046
4,quantity,12616
5,part,8204
6,for,6119
7,name,5604
8,and,5471
9,hazmat,4889


In [10]:
# Optional manual synonyms for unknown classification.
# Add high-frequency patterns discovered above.
CONTAINER_TYPE_SYNONYMS = {
    # pattern : canonical_container_type
    "erlenmeyer": "flask",
    "round bottom": "flask",
    "volumetric flask": "flask",
    "volflask": "flask",
    "media bottle": "bottle",
    "reagent bottle": "bottle",
    "cryovial": "vial",
    "centrifuge tube": "tube",
    "petri": "dish",
    "boston round": "bottle",
    "packer": "bottle",
    "french square": "bottle",
    "straight sided round": "bottle",
    "wide mouth round": "bottle",
    "medium round": "bottle",
    "handled round jug": "bottle",
    "jug": "bottle", 
    "desiccator": "jar",
    "standard wide mouth": "bottle",
    "precision point": "bottle",
    "aerosol": "bottle",  # or "spray_bottle" if you want finer subtype

    # Newly observed unknown-pattern themes
    "solution reservoir": "reservoir",
    "reagent reservoir": "reservoir",
    "reservoir": "reservoir",
    "ringcaps": "cap",
    "sealing film": "closure",
    " seal": "closure",
    "o-ring": "closure",
    "sterilization pouch": "pouch",
    "sterilization pouches": "pouch",
    "sampling bags": "bag",
    "specimen transport": "bag",
    "microtube storage box": "box",
    "storage box": "box",
    "weighing boats": "boat",
    "plastic cups": "cup",
    "strainer": "filter",
    "filter": "filter",

    # Chromatography products that often miss explicit 'column'
    "hypersil": "column",
    " c18": "column",
    " c8": "column",
    " pfp": "column",
    " pah": "column",
    "phenyl": "column",
    "hplc": "column",
    "prepcolumn": "column",
    "chx col": "column",

}


# Keep this OFF by default: cluster labels can be noisy / impure.
USE_LABEL_FAMILY_FALLBACK = False

LABEL_FAMILY_FALLBACK = {
    "Caps and Closures": "closure",
    "Chromatography Columns": "column",
    "Laboratory Bottles": "bottle",
    "Tubes and Vials": "tube",
}


def detect_container_type_with_synonyms(
    text: str,
    spec_norm: dict | None = None,
    proposed_label: str | None = None,
) -> str | None:
    # 1) existing deterministic detector
    base = detect_container_type(text, spec_norm)
    if base is not None:
        return base

    # 2) synonym fallback
    low = normalize_text(text)
    for pattern, mapped in CONTAINER_TYPE_SYNONYMS.items():
        if pattern in low:
            return mapped

    # 3) optional label-informed fallback for persistently ambiguous rows
    if USE_LABEL_FAMILY_FALLBACK and proposed_label in LABEL_FAMILY_FALLBACK:
        return LABEL_FAMILY_FALLBACK[proposed_label]
    return None


# Reclassify using synonym-extended detector
print(f"USE_LABEL_FAMILY_FALLBACK={USE_LABEL_FAMILY_FALLBACK}")
before_unknown = int(df["CONTAINER_FAMILY"].isna().sum())
df["CONTAINER_FAMILY"] = [
    detect_container_type_with_synonyms(t, s, lbl)
    for t, s, lbl in zip(df["NORM_TEXT"], df["SPEC_NORM"], df["PROPOSED_LABEL"])
]
df["CONTAINER_TYPE"] = df["CONTAINER_FAMILY"]
after_unknown = int(df["CONTAINER_FAMILY"].isna().sum())

print(f"Unknown before: {before_unknown:,}")
print(f"Unknown after : {after_unknown:,}")
print(f"Recovered      : {before_unknown - after_unknown:,}")

display(
    df["CONTAINER_FAMILY"].fillna("UNKNOWN").value_counts().rename_axis("container_type").reset_index(name="count")
)

print("\nUnknown rate by PROPOSED_LABEL:")
unknown_by_label = (
    df.assign(_is_unknown=df["CONTAINER_FAMILY"].isna())
    .groupby("PROPOSED_LABEL", dropna=False)
    .agg(rows=("PRODUCT_ID", "count"), unknown=("_is_unknown", "sum"))
    .reset_index()
)
unknown_by_label["unknown_pct"] = (100.0 * unknown_by_label["unknown"] / unknown_by_label["rows"]).round(2)
display(unknown_by_label.sort_values("unknown_pct", ascending=False))

USE_LABEL_FAMILY_FALLBACK=False
Unknown before: 18,492
Unknown after : 15,210
Recovered      : 3,282


,container_type,count
0,UNKNOWN,15210
1,bottle,11581
2,flask,11068
3,tube,10410
4,vial,7638
5,cap,6835
6,column,4080
7,beaker,2359
8,cylinder,1621
9,closure,1514



Unknown rate by PROPOSED_LABEL:


,PROPOSED_LABEL,rows,unknown,unknown_pct
0,Caps and Closures,12894,5292,41.04
1,Chemical Safety,12833,3067,23.90
3,Laboratory Bottles,13605,2373,17.44
6,Tubes and Vials,14298,2089,14.61
5,Pyrex Glassware,6989,708,10.13
4,Laboratory Glassware,14815,1472,9.94
2,Chromatography Columns,3577,209,5.84


After adjusting `CONTAINER_TYPE_SYNONYMS`, rerun the canonical grouping cells (`Build canonical identity key + IDs` and `Select representative record`) before saving outputs.

In [11]:
# Unique container_type values + counts
container_summary = (
    df["CONTAINER_TYPE"]
    .fillna("UNKNOWN")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("CONTAINER_TYPE")
    .reset_index(name="COUNT")
)

display(container_summary)

,CONTAINER_TYPE,COUNT
0,UNKNOWN,15210
1,bottle,11581
2,flask,11068
3,tube,10410
4,vial,7638
5,cap,6835
6,column,4080
7,beaker,2359
8,cylinder,1621
9,closure,1514


In [12]:
# ------------------
# Save outputs locally
# ------------------
canonical_csv = OUT_DIR / "canonical_glassware_products.csv"
membership_csv = OUT_DIR / "canonical_glassware_membership.csv"
summary_json = OUT_DIR / "canonical_glassware_summary.json"

canonical_df.to_csv(canonical_csv, index=False)
membership_df.to_csv(membership_csv, index=False)

summary = {
    "source_products_table": SOURCE_PRODUCTS_TABLE,
    "source_labels_table": SOURCE_LABELS_TABLE,
    "target_labels": TARGET_LABELS,
    "rows_input": int(len(df)),
    "rows_canonical": int(len(canonical_df)),
    "rows_membership": int(len(membership_df)),
    "groups_needing_review": int(canonical_df["NEEDS_REVIEW"].sum()) if "NEEDS_REVIEW" in canonical_df.columns else None,
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:")
print("-", canonical_csv)
print("-", membership_csv)
print("-", summary_json)
summary

Saved:
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_products.csv
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_membership.csv
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_summary.json


{'source_products_table': 'SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LCG',
 'source_labels_table': 'SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PROPOSED_L5',
 'target_labels': ['Laboratory Bottles',
  'Caps and Closures',
  'Laboratory Glassware',
  'Tubes and Vials',
  'Pyrex Glassware',
  'Chemical Safety',
  'Chromatography Columns'],
 'rows_input': 79011,
 'rows_canonical': 21702,
 'rows_membership': 79011,
 'groups_needing_review': 11163}

In [13]:
# ------------------
# Optional write-back to Snowflake
# ------------------
if WRITE_TO_SNOWFLAKE:
    sf.sql(f"USE DATABASE {DB}").collect()
    sf.sql(f"USE SCHEMA {SCHEMA}").collect()

    sf_can = sf.create_dataframe(canonical_df)
    sf_mem = sf.create_dataframe(membership_df)

    sf_can.write.mode("overwrite").save_as_table(CANONICAL_TABLE)
    sf_mem.write.mode("overwrite").save_as_table(MEMBERSHIP_TABLE)

    print(f"Wrote: {CANONICAL_TABLE}")
    print(f"Wrote: {MEMBERSHIP_TABLE}")
else:
    print("Set WRITE_TO_SNOWFLAKE=True to publish canonical tables.")

Set WRITE_TO_SNOWFLAKE=True to publish canonical tables.
